# Intermediate portfolio selection with a variational quantum model

Same objective as the beginner notebook, on a larger instance. Choose exactly $K$ of $n$ assets to maximize

$$
f(x)=\sum_i \mu_i x_i - q \sum_{i,j}\sigma_{ij} x_i x_j, \qquad \sum_i x_i = K.
$$

This notebook takes the **quantum machine learning** route, not a deeper QAOA. A parameterized circuit is trained so that its measurement distribution prefers high-$f$ portfolios of weight $K$. The circuit uses **12 qubits**. The device it is submitted to, `ibm_basquecountry`, has **156 qubits**; the extra qubits stay idle. Training calls the circuit hundreds of times, so those calls use a noiseless statevector. One trained circuit is sampled on the device.

| Piece | What it is |
|---|---|
| Model | Feature map from each asset's marginal score, then two layers of entangling gates and trainable rotations |
| Training data | Every feasible portfolio, scored by the known objective $f$ |
| Loss | $-\sum_{\|x\|_1=K} P_\theta(x)\, e^{\alpha f(x)}$ |

Learning buys the step from the product distribution (which recovers a greedy ranking) to a distribution whose most likely legal portfolio accounts for covariance. It does not buy a hidden label: $f$ is classical and is the training signal.

## 1. Portfolio and classical baselines

$n=12$, $K=4$, so there are $\binom{12}{4}=495$ feasible portfolios. Brute force is still exact and is used only as the test score.

The covariance is one-factor, $\sigma_{ij}=0.03\,\beta_i\beta_j+\delta_{ij}\omega_i^2$. Assets with a high $\beta$ look attractive on return and expensive once they are held together.

Two classical baselines:

- **Marginal greedy.** Score asset $i$ by $\mu_i-q\sigma_{ii}$ and keep the top $K$. This ignores $\sigma_{ij}$ for $i\neq j$.
- **Swap search.** From that portfolio, exchange one held asset for one excluded asset whenever $f$ improves.

In [1]:
import numpy as np
from itertools import combinations

n, K, q = 12, 4, 2.0
mu = np.array([0.08, 0.18, 0.22, 0.11, 0.07, 0.19, 0.13, 0.16, 0.09, 0.21, 0.12, 0.15])
beta = np.array([0.6, 1.3, 1.6, 0.5, 0.4, 1.4, 0.8, 1.1, 0.7, 1.5, 0.9, 1.0])
idio = np.array([0.04, 0.06, 0.14, 0.04, 0.03, 0.11, 0.05, 0.07, 0.04, 0.12, 0.05, 0.06])
sigma = 0.03 * np.outer(beta, beta) + np.diag(idio ** 2)

def portfolio_value(bits):
    x = np.asarray(bits, dtype=float)
    return float(x @ mu - q * x @ sigma @ x)

feasible = {}
for chosen in combinations(range(n), K):
    x = np.zeros(n)
    x[list(chosen)] = 1.0
    feasible[chosen] = portfolio_value(x)

optimal = max(feasible, key=feasible.get)
f_star = feasible[optimal]

marginal_score = mu - q * np.diag(sigma)
greedy = tuple(sorted(np.argsort(-marginal_score)[:K].tolist()))

current = set(greedy)
current_value = feasible[tuple(sorted(current))]
improved = True
while improved:
    improved = False
    best_set, best_value = set(current), current_value
    for held in list(current):
        for candidate in range(n):
            if candidate in current:
                continue
            trial = set(current)
            trial.remove(held)
            trial.add(candidate)
            value = feasible[tuple(sorted(trial))]
            if value > best_value + 1e-15:
                best_set, best_value, improved = trial, value, True
    current, current_value = best_set, best_value
swap = tuple(sorted(current))

print(f"feasible portfolios : {len(feasible)}")
print(f"brute-force optimum : assets {list(optimal)}  f* = {f_star:.6f}")
print(f"marginal greedy     : assets {list(greedy)}  f = {feasible[greedy]:.6f}  ratio = {feasible[greedy] / f_star:.3f}")
print(f"swap search         : assets {list(swap)}  f = {feasible[swap]:.6f}  ratio = {feasible[swap] / f_star:.3f}")
assert len(optimal) == K
assert np.linalg.eigvalsh(sigma).min() > 0
assert f_star > 0

feasible portfolios : 495
brute-force optimum : assets [0, 3, 4, 6]  f* = 0.059400
marginal greedy     : assets [3, 6, 7, 11]  f = -0.168800  ratio = -2.842
swap search         : assets [0, 3, 4, 6]  f = 0.059400  ratio = 1.000


## 2. The circuit

Qubit $i$ is asset $i$. The first gate is fixed by the marginal score, scaled into $(0.2\pi,\, 0.8\pi)$:

$$
R_Y(\phi)|0\rangle = \cos(\phi/2)\,|0\rangle + \sin(\phi/2)\,|1\rangle,
\qquad P(x_i=1)=\sin^2(\phi_i/2).
$$

At $\theta=0$ the later rotations are the identity and the $\mathrm{CZ}$ line does not change computational-basis probabilities, so the most likely weight-$K$ string is the marginal greedy portfolio. Each trained layer is that $\mathrm{CZ}$ line followed by $R_Y R_Z$. The $R_Y$ is what can turn a correlation into a different portfolio.


In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

LAYERS = 2
score_unit = (marginal_score - marginal_score.min()) / (marginal_score.max() - marginal_score.min())
feature_angle = np.pi * (0.2 + 0.6 * score_unit)

def circuit(theta):
    angles = np.asarray(theta, dtype=float).reshape(LAYERS, n, 2)
    qc = QuantumCircuit(n)
    for qubit in range(n):
        qc.ry(feature_angle[qubit], qubit)
    for layer in range(LAYERS):
        for qubit in range(n - 1):
            qc.cz(qubit, qubit + 1)
        for qubit in range(n):
            qc.ry(float(angles[layer, qubit, 0]), qubit)
            qc.rz(float(angles[layer, qubit, 1]), qubit)
    return qc

def probabilities(theta):
    return np.asarray(Statevector(circuit(theta)).probabilities(), dtype=float)

def most_likely_legal(probs):
    best_key, best_prob = None, -1.0
    for chosen in feasible:
        index = sum(1 << qubit for qubit in chosen)
        if probs[index] > best_prob:
            best_key, best_prob = chosen, float(probs[index])
    return best_key, best_prob

initial = most_likely_legal(probabilities(np.zeros(LAYERS * n * 2)))
print(f"qubits in the model : {n}")
print(f"parameters          : {LAYERS * n * 2}")
print(f"theta = 0 mode      : assets {list(initial[0])}  f = {feasible[initial[0]]:.6f}")
assert initial[0] == greedy


qubits in the model : 12
parameters          : 48
theta = 0 mode      : assets [3, 6, 7, 11]  f = -0.168800


## 3. Loss and training

Every feasible portfolio is training data. Its label is its own $f$, not a flag saying whether it is optimal. Illegal bitstrings contribute nothing:

$$
L(\theta) = -\sum_{\|x\|_1=K} P_\theta(x)\, e^{\alpha f(x)}, \qquad \alpha = 25.
$$

The exponential is there because a raw shift $f-f_{\min}$ makes a mediocre portfolio almost as valuable as the best one, and the optimizer then piles probability on the mediocre set. $\alpha=25$ makes $e^{\alpha f}$ decay as soon as $f$ drops below the optimum. Subtracting a constant inside the exponential would only multiply $L$ by a positive constant, so centring on $f^\star$ is unnecessary.

COBYLA is local. The run keeps the best of a zero start and five random starts. The reported portfolio is the legal bitstring with the largest probability, scored with $f$.

In [3]:
from scipy.optimize import minimize

ALPHA = 25.0
reward = np.zeros(1 << n)
for chosen, value in feasible.items():
    reward[sum(1 << i for i in chosen)] = np.exp(ALPHA * value)

def loss(theta):
    return -float(probabilities(theta) @ reward)

rng = np.random.default_rng(0)
starts = [np.zeros(LAYERS * n * 2)]
starts += [rng.normal(0.0, 0.35, LAYERS * n * 2) for _ in range(5)]

trained, trained_loss = None, np.inf
for start in starts:
    result = minimize(loss, start, method="COBYLA", options={"maxiter": 120, "rhobeg": 0.35})
    if result.fun < trained_loss:
        trained, trained_loss = result.x.copy(), float(result.fun)

probs = probabilities(trained)
chosen, mode_prob = most_likely_legal(probs)
legal_index = np.array([sum(1 << i for i in c) for c in feasible])
legal_mass = float(probs[legal_index].sum())
mean_legal_f = float(probs[legal_index] @ np.array(list(feasible.values()))) / legal_mass

print(f"loss                : {trained_loss:.6f}")
print(f"probability on K=4  : {legal_mass:.3f}")
print(f"P(mode)             : {mode_prob:.3f}")
print(f"QML portfolio       : assets {list(chosen)}  f = {feasible[chosen]:.6f}  ratio = {feasible[chosen] / f_star:.3f}")
print(f"mean legal f / f*   : {mean_legal_f / f_star:.3f}")
print(f"marginal ratio      : {feasible[greedy] / f_star:.3f}")
print(f"swap ratio          : {feasible[swap] / f_star:.3f}")
assert chosen == optimal

loss                : -1.774348
probability on K=4  : 0.463
P(mode)             : 0.384
QML portfolio       : assets [0, 3, 4, 6]  f = 0.059400  ratio = 1.000
mean legal f / f*   : 0.433
marginal ratio      : -2.842
swap ratio          : 1.000


## 4. The same circuit on `ibm_basquecountry`

Training stays on the statevector. The device samples the trained circuit once: job `dbm6o2e0fc9c73dhnev0`, 4096 shots, 12 qubits routed onto the 156-qubit Heron. This cell reads that job. It does not submit another one.


In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService

SHOTS = 4096
finished = QiskitRuntimeService(channel="ibm_quantum_platform").job("dbm6o2e0fc9c73dhnev0")
print(finished.status())
counts = finished.result()[0].data.meas.get_counts()
legal_counts = {}
used = 0
for bitstring, count in counts.items():
    bits = bitstring[::-1]  # rightmost character is qubit 0
    assets = tuple(i for i in range(n) if bits[i] == "1")
    if len(assets) != K:
        continue
    used += count
    legal_counts[assets] = legal_counts.get(assets, 0) + count
hardware_best = max(legal_counts, key=lambda assets: feasible[assets])
hardware_mode = max(legal_counts, key=legal_counts.get)
print(f"device shots on weight K : {used} / {SHOTS}  ({used / SHOTS:.3f})")
print(f"best device sample       : assets {list(hardware_best)}  f = {feasible[hardware_best]:.6f}  ratio = {feasible[hardware_best] / f_star:.3f}")
print(f"most frequent legal      : assets {list(hardware_mode)}  shots = {legal_counts[hardware_mode]}")
print(f"noiseless QML mode       : assets {list(chosen)}  ratio = {feasible[chosen] / f_star:.3f}")


DONE
device shots on weight K : 1697 / 4096  (0.414)
best device sample       : assets [0, 3, 4, 6]  f = 0.059400  ratio = 1.000
most frequent legal      : assets [0, 3, 4, 6]  shots = 1281
noiseless QML mode       : assets [0, 3, 4, 6]  ratio = 1.000


## How to read the result

The marginal greedy portfolio is the feature map before training. Swap search is the classical bar. The QML portfolio is the legal bitstring with the largest statevector probability, scored with $f$. Its approximation ratio is $f/f^\star$ over all 495 feasible portfolios.

The device row is the same circuit after routing onto the Heron, scored the same way. The mean legal score, printed next to the mode, averages $f$ over every feasible portfolio that still has amplitude.

This is one 12-qubit circuit on a shared 156-qubit device, plus an exact classical enumeration. It is not a quantum advantage.
